# Monitoring ML Systems

Production ML systems need three monitoring layers: system health (is it up?), data quality (is the input clean?), and model quality (are predictions still good?). This note implements PSI and KS drift detectors and builds a monitoring plan template.

## What Interviewers Test
- The three monitoring layers and what belongs in each
- PSI and KS drift detection implementations
- Delayed-label handling: proxy metrics until labels arrive
- Alert design: thresholds vs anomaly detection
- A monitoring plan you can recite in system design interviews

In [ ]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

def psi(expected, actual, n_bins=10):
    """Population Stability Index. PSI > 0.2 = significant drift."""
    bins = np.linspace(min(expected.min(), actual.min()),
                       max(expected.max(), actual.max()) + 1e-6, n_bins + 1)
    exp_pct = np.histogram(expected, bins)[0] / len(expected)
    act_pct = np.histogram(actual,   bins)[0] / len(actual)
    exp_pct = np.clip(exp_pct, 1e-6, None)
    act_pct = np.clip(act_pct, 1e-6, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

def ks_drift_test(expected, actual, alpha=0.05):
    """Kolmogorov-Smirnov test for distribution shift."""
    stat, pval = stats.ks_2samp(expected, actual)
    return {'statistic': stat, 'p_value': pval, 'drift': pval < alpha}

# Simulate a feature drifting over time
n = 2000
train_dist = np.random.normal(0, 1, n)

drifts = {
    'No drift (t=0)':      np.random.normal(0.0, 1.0, 500),
    'Slight drift (t=1w)': np.random.normal(0.3, 1.1, 500),
    'Moderate (t=1m)':     np.random.normal(0.8, 1.3, 500),
    'Severe (t=3m)':       np.random.normal(2.0, 1.8, 500),
}

print(f"{'Period':<25} {'PSI':>8} {'Status':>12} {'KS p-val':>10}")
print("-" * 60)
for period, dist in drifts.items():
    psi_val = psi(train_dist, dist)
    ks = ks_drift_test(train_dist, dist)
    status = 'OK' if psi_val < 0.1 else ('INVESTIGATE' if psi_val < 0.2 else 'RETRAIN')
    print(f"{period:<25} {psi_val:>8.4f} {status:>12} {ks['p_value']:>10.4f}")


In [ ]:
# --- Prediction distribution monitoring ---
def monitor_prediction_distribution(scores_ref, scores_current, sigma_threshold=2.0):
    """Alert if mean score shifts by more than sigma_threshold std deviations."""
    mean_ref, std_ref = scores_ref.mean(), scores_ref.std()
    mean_cur = scores_current.mean()
    z_score = abs(mean_cur - mean_ref) / (std_ref + 1e-8)
    alert = z_score > sigma_threshold
    return {
        'ref_mean': mean_ref, 'cur_mean': mean_cur,
        'z_score': z_score, 'alert': alert
    }

# Simulate prediction scores before and after model degradation
scores_ref = np.random.beta(2, 8, 5000)          # reference: low scores (mostly negatives)
scores_ok   = np.random.beta(2.1, 7.9, 1000)     # slight variation — OK
scores_drift = np.random.beta(4, 6, 1000)         # shifted — model predicting higher

for label, scores in [('Stable', scores_ok), ('Drifted', scores_drift)]:
    result = monitor_prediction_distribution(scores_ref, scores)
    print(f"{label}: ref_mean={result['ref_mean']:.4f}, cur_mean={result['cur_mean']:.4f}, "
          f"z={result['z_score']:.2f}, alert={result['alert']}")


## Monitoring Plan Template

Recite this in interviews for the "how do you monitor?" question:

| Layer | Metric | Frequency | Alert threshold | Action |
|---|---|---|---|---|
| **System** | Latency p99 | Real-time | > 200ms | Page on-call |
| **System** | Error rate | Real-time | > 0.1% | Page on-call |
| **System** | QPS anomaly | Real-time | > 2× or < 0.5× baseline | Alert |
| **Data** | Feature PSI | Hourly | > 0.2 on any feature | Investigate |
| **Data** | Null rate | Hourly | > 2× baseline | Investigate |
| **Model** | Score mean/std | Hourly | > 2σ shift | Investigate |
| **Model** | Label rate (proxy) | Daily | < 50% expected | Investigate |
| **Business** | Primary metric | Daily | > 5% drop vs 7d avg | A/B review |
| **Business** | Guardrail metric | Daily | Any drop | A/B review |

**Alert fatigue rule:** Pages (immediate action) only for system-level issues. Investigate-level alerts route to a monitoring dashboard + ticket. Suppress repeat alerts for the same root cause.


## Delayed-Label Handling

For systems where true labels arrive late (fraud chargebacks: 30–90 days, user satisfaction: survey response days):

1. **Proxy labels:** Use fast, correlated signals (customer dispute, issuer decline) until true labels arrive
2. **Temporal holdout:** Evaluate on a set where true labels are already available (older cohort)
3. **Label imputation:** Assume negatives for unlabeled examples after a timeout
4. **Monitoring gaps:** Track the gap between prediction volume and labeled volume — a growing gap signals label delay issues


## Common Interview Questions

**Q: What are the three layers of ML monitoring?**
(1) System health: is the serving infrastructure up? latency, error rate, QPS. (2) Data quality: are inputs as expected? feature distributions via PSI, null rates, schema violations. (3) Model/business quality: are predictions still useful? score distribution, offline metrics on labeled holdout, business KPIs. You need all three because each layer can fail independently.

**Q: How does PSI differ from KS test?**
Both detect distributional shift. PSI is an industry-standard threshold-based metric (interpretable categories: 0.1/0.2 cutoffs) commonly used in credit risk; it's asymmetric and uses binning. The KS test produces a p-value for statistical significance and is more sensitive to fine-grained differences. In practice, use PSI for operational thresholds and KS as a complementary check.

**Q: How do you handle monitoring when labels arrive with a 30-day delay?**
Use proxy labels (faster correlates) for near-real-time model quality monitoring. Maintain a fixed evaluation cohort where labels have already arrived and compute offline metrics on it regularly. Set up label volume monitoring — track how many transactions from each date have received labels, and alert if the ratio drops below expected.

**Q: How do you prevent alert fatigue in ML monitoring?**
Reserve pages (immediate wake-up) only for system-critical issues (latency, error rate). Route data/model drift to investigate-level alerts in a monitoring dashboard. Set business metric alerts as daily summaries, not real-time. Suppress duplicate alerts within a time window. Review alert triggers quarterly and tune thresholds based on false-alarm rates.

## Key Takeaways
- Three layers: system (is it up?) → data (clean inputs?) → model (good predictions?)
- PSI: 0.1 stable | 0.1–0.2 investigate | >0.2 retrain; KS test for statistical significance
- Monitor prediction distribution: alert if mean shifts > 2σ from reference
- Delayed labels: proxy metrics for fast feedback; temporal holdout for offline evaluation
- Alert taxonomy: page (system) > investigate (data/model) > dashboard (business)
- Always monitor QPS — sudden drops mean traffic routing issues, not model failures